# Fine-tuning GPT-2 with LoRA using `peft_lib`

This end-to-end tutorial shows the full PEFT workflow:

1. Load a GPT-2 model and wrap it with LoRA.
2. See the **trainable-parameter reduction** vs full fine-tuning.
3. Train only the adapters and plot the loss curve.
4. Profile the **memory** (optimizer-state) savings.
5. **Merge** the adapter for zero-overhead inference and verify equivalence.
6. **Save / load** the (tiny) adapter checkpoint.

> To use the *pretrained* GPT-2 on a *real* dataset, replace the config-built
> model below with `AutoModelForCausalLM.from_pretrained("gpt2")` and feed real
> tokenized batches — everything else stays identical. We build from config here
> so the notebook runs offline.

In [ ]:
import torch
from transformers import GPT2Config, GPT2LMHeadModel

from peft_lib import LoRAConfig, get_peft_model

torch.manual_seed(42)

# A small-but-real GPT-2 architecture (swap for from_pretrained('gpt2') for the real thing).
config = GPT2Config(n_layer=4, n_head=4, n_embd=128, vocab_size=256, n_positions=128)
model = GPT2LMHeadModel(config)
print(model.config.model_type, '|', sum(p.numel() for p in model.parameters()), 'params')

## 1. Wrap with LoRA

`target_modules` is inferred from `config.model_type` (`gpt2` → `c_attn`). LoRA
freezes the backbone and injects zero-initialised low-rank adapters, so the
wrapped model is numerically identical to GPT-2 until we train.

In [ ]:
peft_model = get_peft_model(model, LoRAConfig(r=8, alpha=16, dropout=0.05))
peft_model.print_trainable_parameters()
trainable, total = peft_model.get_nb_trainable_parameters()
print(f'Full fine-tuning would train {total:,} params; LoRA trains {trainable:,} '
      f'({100 * trainable / total:.2f}%) — a {total / trainable:.0f}x reduction.')

## 2. A toy language-modelling dataset

We make a small set of integer-token sequences. The default trainer loss is
position-wise cross-entropy; for next-token LM we pass `causal_lm_loss`.

In [ ]:
torch.manual_seed(0)
vocab, seq, batch = config.vocab_size, 32, 8

def make_batches(n_batches):
    batches = []
    for _ in range(n_batches):
        ids = torch.randint(0, vocab, (batch, seq))
        batches.append({'input_ids': ids, 'labels': ids})
    return batches

train_data = make_batches(30)
eval_data = make_batches(4)
len(train_data), train_data[0]['input_ids'].shape

## 3. Train the adapters

In [ ]:
from peft_lib.training import PEFTTrainer, TrainableParamLogger, TrainerConfig, causal_lm_loss

cfg = TrainerConfig(
    learning_rate=3e-3,
    num_epochs=4,
    scheduler='cosine',
    warmup_ratio=0.1,
    logging_steps=5,
    device='cpu',
)
trainer = PEFTTrainer(
    peft_model, cfg, train_data, eval_dataloader=eval_data,
    compute_loss=causal_lm_loss, callbacks=[TrainableParamLogger()],
)
history = trainer.train()
print('final eval:', trainer.evaluate())

## 4. Plot the loss curve

In [ ]:
losses = [(h['step'], h['loss']) for h in history if 'loss' in h]
print(f'loss: {losses[0][1]:.3f} -> {losses[-1][1]:.3f}')
try:
    import matplotlib.pyplot as plt
    steps, vals = zip(*losses)
    plt.figure(figsize=(6, 3))
    plt.plot(steps, vals, marker='o', ms=3)
    plt.xlabel('optimizer step'); plt.ylabel('train loss'); plt.title('LoRA fine-tuning')
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
except ImportError:
    print('(install matplotlib to see the curve)')

## 5. Memory profile: optimizer-state savings

The big VRAM win from PEFT is the optimizer state (Adam keeps 2 fp32 moments per
*trainable* parameter).

In [ ]:
from peft_lib.benchmarks import lora_flop_overhead_ratio, optimizer_state_bytes

lora_opt = optimizer_state_bytes(peft_model)
full_opt = sum(p.numel() for p in model.parameters()) * 8
print(f'Adam state: LoRA {lora_opt / 1e3:.1f} KB vs full {full_opt / 1e6:.1f} MB '
      f'({full_opt / lora_opt:.0f}x smaller)')
print(f'Forward FLOP overhead at (D=768, r=8): {lora_flop_overhead_ratio(768, 768, 8):.2%}')

## 6. Merge for zero-overhead inference

`merge_and_unload` folds `s·BA` into the GPT-2 weights and returns the plain
model — numerically equivalent, with no adapter overhead at inference.

In [ ]:
peft_model.eval()
probe = torch.randint(0, vocab, (2, seq))
with torch.no_grad():
    before = peft_model(input_ids=probe).logits
merged = peft_model.merge_and_unload()
with torch.no_grad():
    after = merged(input_ids=probe).logits
print('merge equivalent:', torch.allclose(before, after, atol=1e-4))
print('type after unload:', type(merged).__name__)

## 7. Save & load the adapter

The checkpoint contains only the adapter weights + config — kilobytes, not the
whole model.

In [ ]:
import os, tempfile

from peft_lib import LoRAModel

# Re-wrap a fresh model (merge_and_unload consumed the previous wrapper).
torch.manual_seed(42)
fresh = GPT2LMHeadModel(config)
p2 = get_peft_model(fresh, LoRAConfig(r=8, alpha=16, dropout=0.05))
with tempfile.TemporaryDirectory() as d:
    p2.save_pretrained(d)
    print('checkpoint files:', sorted(os.listdir(d)))
    size_kb = os.path.getsize(os.path.join(d, 'adapter_model.safetensors')) / 1e3
    print(f'adapter size: {size_kb:.1f} KB')
    torch.manual_seed(42)
    reloaded = LoRAModel.from_pretrained(GPT2LMHeadModel(config), d)
    print('reloaded:', type(reloaded).__name__)

## Recap

* LoRA trained a tiny fraction of GPT-2's parameters and the loss dropped.
* Optimizer memory shrank by orders of magnitude.
* `merge_and_unload` gave a numerically-equivalent, overhead-free model.
* The adapter checkpoint is kilobytes.

Swap `LoRAConfig` for `DoRAConfig`, `VeRAConfig`, `IA3Config`, … to try other
methods — the API is identical.